In [0]:
%run ../00-common/01.environment-config

In [0]:
target_table = f"{catlog_name}.{gold_schema}.results_fact"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df = (
    spark.table(f"{catlog_name}.{silver_schema}.results")
        .withColumn("session_type", F.lit("RACE"))
        .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
    )


In [0]:
display(results_df)

In [0]:
sprints_df = (
    spark.table(f"{catlog_name}.{silver_schema}.sprints")
        .withColumn("session_type", F.lit("SPRINT"))
        .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
    )

In [0]:
results_sprints_df = results_df.union(sprints_df)

In [0]:
results_sprints_df.printSchema()

In [0]:
fact_session_result_df = (
    results_sprints_df
        .withColumn("is_win", F.col("finish_position") == 1)
        .withColumn("is_podium", F.col("finish_position").between(1,3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
display(fact_session_result_df)

In [0]:
(
    fact_session_result_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)